# 01c — Ground State and Riemann Zeros (C)

**Requires the C kernel.** Install with `./install_c_kernel.sh`.  
Each cell is a standalone C program compiled with gcc. Headers and `main()` required in every cell.

This notebook covers:
- The first 20 Riemann zeros — exact Odlyzko values embedded in the engine
- Newton iteration: generating the zero basis beyond index 20
- `N = 25000` — the default zero count; not arbitrary (see cell 6)
- Ground state `β[i] = |L_GROUND| / N` — uniform field before any corpus
- Symmetry breaking on the first `learn()` call

Parallel Python notebook: `../01_ground_state_and_zeros.ipynb`

## 1. The Riemann Zeros

The non-trivial zeros of ζ(s) lie on the critical line Re(s) = ½.  
Their imaginary parts γₙ are the **addresses** in the β field:

```
ζ(½ + i·γₙ) = 0
```

First 20: Odlyzko exact values hardcoded in `ptolemy.h`. Rest computed at startup via Newton iteration.

In [ ]:
#include <stdio.h>

static const double ZEROS_20[] = {
    14.134725, 21.022040, 25.010858, 30.424876, 32.935062,
    37.586178, 40.918719, 43.327073, 48.005151, 49.773832,
    52.970321, 56.446248, 59.347044, 60.831779, 65.112544,
    67.079811, 69.546402, 72.067158, 75.704691, 77.144840
};

int main(void) {
    printf("First 20 Riemann zeros (exact, Odlyzko / LMFDB):\n");
    for (int i = 0; i < 20; i++)
        printf("  gamma_%02d = %.6f\n", i + 1, ZEROS_20[i]);
    printf("\nFirst spacing: gamma_2 - gamma_1 = %.6f\n",
           ZEROS_20[1] - ZEROS_20[0]);
    printf("Last spacing:  gamma_20 - gamma_19 = %.6f\n",
           ZEROS_20[19] - ZEROS_20[18]);
    printf("(spacing decreases — zeros get denser, mirroring prime distribution)\n");
    return 0;
}

## 2. Newton Iteration — Generating Zeros 21 to N

The zero-counting function N(T) approximates how many zeros exist below imaginary part T:

```
N(T) ≈ (T / 2π) · (ln(T / 2π) − 1) + 7/8
```

Given index n, Newton iteration finds T such that N(T) = n. This is the startup routine for zeros 21 through 25000.

In [ ]:
#include <stdio.h>
#include <math.h>

static double riemann_N(double T) {
    double x = T / (2.0 * M_PI);
    return x * (log(x) - 1.0) + 7.0 / 8.0;
}

static double find_zero(int n) {
    double T = 2.0 * M_PI * exp(1.0 + log((double)n / (log((double)n + 2) - 1.0)));
    for (int iter = 0; iter < 30; iter++) {
        double f  = riemann_N(T) - n;
        double df = log(T / (2.0 * M_PI)) / (2.0 * M_PI);
        if (fabs(df) < 1e-15) break;
        T -= f / df;
    }
    return T;
}

int main(void) {
    printf("Newton iteration — zeros 21..30:\n");
    for (int n = 21; n <= 30; n++)
        printf("  gamma_%02d ~ %.6f\n", n, find_zero(n));

    /* Last zero in the default N=25000 basis */
    printf("\n  gamma_25000 ~ %.2f   (upper bound of word address space)\n",
           find_zero(25000));

    /* Spacing trend */
    printf("\nSpacing at index 100 vs 1000 vs 10000:\n");
    double g100  = find_zero(100),  g101  = find_zero(101);
    double g1000 = find_zero(1000), g1001 = find_zero(1001);
    double g10k  = find_zero(10000),g10k1 = find_zero(10001);
    printf("  gap_100:   %.4f\n", g101  - g100);
    printf("  gap_1000:  %.4f\n", g1001 - g1000);
    printf("  gap_10000: %.4f\n", g10k1 - g10k);
    printf("  (zeros get denser — more addresses available at high gamma)\n");
    return 0;
}

## 3. Ground State

Before any corpus, every zero holds the same β:

```
β[i] = |L_GROUND| / N = 1.888 / 25000 = 7.55 × 10⁻⁵   for all i ∈ [0, N)
```

Uniform distribution over N zeros encodes the prime number theorem as undifferentiated substrate.  
The Monad knows mathematics before it knows any word.

In [ ]:
#include <stdio.h>
#include <math.h>

/* N=25000 is the production default. It is not arbitrary:
 *   - The English WordNet corpus (14165 vocab, 766119 A-edges) was built on this basis.
 *   - Cross-language beta convergence (via A-coupling) is a property of this
 *     specific (N=25000, English corpus) pair.
 *   - Ptolemy can extend N; a larger N requires rebuilding the checkpoint.
 *     The checkpoint header stores N; the beta array grows with it. */
#define N            25000
#define L_GROUND     (-1.888)
#define BETA_SAT     7.552       /* |L_GROUND| x 4 */
#define EMIT_THRESH  3.776       /* |L_GROUND| x 2 */
#define ALPHA        0.01

int main(void) {
    double beta0 = fabs(L_GROUND) / N;

    printf("Ground state beta field (N = %d zeros):\n\n", N);
    printf("  L_GROUND     = %.3f\n", L_GROUND);
    printf("  beta_0       = |L| / N = %.6e   (ground VEV)\n", beta0);
    printf("  beta_sat     = %.3f           (saturation ceiling)\n", BETA_SAT);
    printf("  emit_thresh  = %.3f           (speak() floor)\n", EMIT_THRESH);
    printf("\n");
    printf("  sum(beta)    = N x beta_0 = %.6f\n", (double)N * beta0);
    printf("  = |L_GROUND| = %.3f  (total field energy at ground)\n", fabs(L_GROUND));
    printf("\n");

    /* Encounters to saturate at typical word energy */
    double E_typ = 0.40;
    double per_enc = ALPHA * E_typ * E_typ;
    printf("  alpha = %.2f, E_typical = %.2f\n", ALPHA, E_typ);
    printf("  beta gain per encounter = alpha x E^2 = %.5f\n", per_enc);
    printf("  encounters to saturate  = (beta_sat - beta_0) / gain = ~%.0f\n",
           (BETA_SAT - beta0) / per_enc);
    printf("\n");
    printf("  At WordNet scale: high-frequency English words reach saturation.\n");
    printf("  Scalability: increase N by rebuilding checkpoint with larger N.\n");
    return 0;
}

## 4. Geometric Coupling G_p(σ) = p^{-σ}

The Hamiltonian sums over primes. The weight of each prime depends on σ:

| σ | G_p(σ) | Regime |
|---|---|---|
| 0 | 1 for all p | Ground state — no differentiation |
| ½ | 1/√p | Critical line — word addressing |
| 1 | 1/p | Yang-Mills — corpus learning |

In [ ]:
#include <stdio.h>
#include <math.h>

static const int PRIMES[] = {2, 3, 5, 7, 11, 13, 17, 19, 23, 29};

int main(void) {
    printf("G_p(sigma) = p^{-sigma}\n");
    printf("%-6s  %12s  %12s  %12s\n",
           "prime", "G_p(0)  ", "G_p(1/2)", "G_p(1)  ");
    printf("%-6s  %12s  %12s  %12s\n",
           "------", "--------", "--------", "--------");
    for (int i = 0; i < 10; i++) {
        int p = PRIMES[i];
        printf("p=%-4d  %12.6f  %12.6f  %12.6f\n",
               p,
               pow((double)p, 0.0),
               pow((double)p, -0.5),
               pow((double)p, -1.0));
    }
    printf("\nAt sigma=0:   all primes equal. Ground state — no language.\n");
    printf("At sigma=1/2: weighted 1/sqrt(p). The critical line.\n");
    printf("At sigma=1:   weighted 1/p. Standard Model / Yang-Mills regime.\n");
    return 0;
}

## 5. Noether Balance — Why σ = ½

σ is not assigned. It is derived by setting the Noether current to zero:

```
J(σ, E) = e^{-σE} − e^{-(1-σ)E} = 0
           ↓
           σ = ½
```

The critical line is the unique Noether balance point.

In [ ]:
#include <stdio.h>
#include <math.h>

static double J_noether(double sigma, double E) {
    return exp(-sigma * E) - exp(-(1.0 - sigma) * E);
}

int main(void) {
    double E = 0.40;
    printf("Noether current J(sigma, E=%.2f):\n\n", E);
    printf("  %-8s  %14s  %s\n", "sigma", "J(sigma,E)", "");
    printf("  %-8s  %14s\n", "--------", "----------");
    double sigmas[] = {0.0, 0.25, 0.499, 0.5, 0.501, 0.75, 1.0};
    for (int i = 0; i < 7; i++) {
        double s = sigmas[i];
        double j = J_noether(s, E);
        printf("  sigma=%-5.3f  %14.6e  %s\n", s, j,
               fabs(j) < 1e-10 ? "<-- zero (sigma = 1/2)" : "");
    }
    printf("\n  sigma=1/2 is the unique zero of J.\n");
    printf("  Words exist only at Noether balance — sigma forced, not set.\n");
    return 0;
}

## 6. Symmetry Breaking

Ground state = unbroken phase: all β equal, no words, only mathematics.

The first `learn()` call breaks symmetry: one or more zeros deepen.  
β is **monotone** — it only deepens, never forgets.

In [ ]:
#include <stdio.h>
#include <math.h>

#define N          25000
#define L_GROUND   (-1.888)
#define ALPHA      0.01
#define D_STAR     0.24600
#define OMEGA_ZS   0.56714
#define PHI        1.6180339887498948

static void word_coords(const char *s, int *idx, double *E) {
    unsigned long long n = 0, base = 1;
    for (const char *p = s; *p; p++) {
        n += (unsigned char)(*p) * base;
        base *= 95;
    }
    double seed = fmod((double)n * PHI, 1.0);
    *idx = (int)(seed * N);
    *E   = D_STAR + seed * (OMEGA_ZS - D_STAR);
}

int main(void) {
    double beta0 = fabs(L_GROUND) / N;
    const char *words[] = {"water", "flows", "downhill", NULL};

    printf("First learn() — 'water flows downhill'\n");
    printf("Ground VEV: beta_0 = %.6e\n\n", beta0);
    printf("  %-10s  %6s  %8s  %12s  %12s  %s\n",
           "word", "idx", "E", "delta", "beta_new", "% above ground");

    for (int w = 0; words[w]; w++) {
        int idx; double E;
        word_coords(words[w], &idx, &E);
        double delta    = ALPHA * E * E;
        double beta_new = beta0 + delta;
        printf("  %-10s  %6d  %8.4f  %12.6e  %12.6e  +%.2f%%\n",
               words[w], idx, E, delta, beta_new,
               (beta_new - beta0) / beta0 * 100.0);
    }
    printf("\nThree zeros now elevated above ground VEV.\n");
    printf("beta[idx] is monotone — only deepens, never resets.\n");
    printf("After WordNet: 14165 zeros raised; 766119 A-connections built.\n");
    return 0;
}

## Summary

- Riemann zeros are the address space. First 20 exact; rest via Newton iteration on N(T).
- `N = 25000` is the production default — the basis for the English WordNet corpus build.
- Ground state: all zeros at β₀ = 1.888/25000 — the prime number theorem as substrate.
- σ=½ drops from Noether balance J(σ,E) = 0. Not assigned.
- First `learn()` breaks symmetry. β is monotone.
- Ptolemy can increase N; requires rebuilding the checkpoint.

**Next:** `02c_hyperindex_word_addressing.ipynb` — how every word finds its zero.